# Q&A notebook

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

┌─────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│          name           │                                                                                  description                                                                                   │
│         varchar         │                                                                                    varchar                                                                                     │
├─────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ D_bacteriaHits          │ This table contains all the MOTUS hits obtained, regardless of their status. It contains the library where the detection took place, taxid, scientific n

## How many virus detections were reported in McLeish2024?

In [2]:

len(db.conn.sql('SELECT * FROM D_virusHits').df()) # 1616 hits in our database

1616

## How many taxa?

In [3]:
len(db.conn.sql('SELECT * FROM D_virusHits').df()['scientific_name'].unique()) # 1616 hits in our database

158

## Did all the PAB bacteria detected correspond to at least a virus in their library?

In [4]:
virus = db.conn.sql('SELECT * FROM D_virusHits').df()
bacteria = db.conn.sql('SELECT * FROM D_pabHits').df()
virus_positive_libraries = virus['library'].unique()
bacteria['is_in_virus'] = bacteria['library'].isin(virus_positive_libraries)
bacteria.groupby(['scientific_name'], as_index=False)['is_in_virus'].sum().query('is_in_virus != 0')

,scientific_name,is_in_virus
0,Achromobacter xylosoxidans,2
1,Acidovorax sp. Leaf160,1
2,Acidovorax sp. Leaf78,1
3,Acinetobacter baumannii,1
4,Acinetobacter pittii,3
...,...,...
122,Stutzerimonas stutzeri,1
123,Variovorax paradoxus,2
124,Xanthomonas campestris,6
125,Xaviernesmea rhizosphaerae,1


In [5]:
len(bacteria.taxid.unique())

127

In [6]:
bacteria.query('taxid == 964')

,library,taxid,scientific_name,is_pab,pab_type,is_in_virus
39,PV026,964,Herbaspirillum seropedicae,True,pab_unknown,False


In [7]:
virus.query('library == "PV026"')

,library,acronym,taxid,scientific_name


In [8]:
virus.library.sort_values().unique()

array(['PV001', 'PV002', 'PV003', 'PV003bgi', 'PV004bgi', 'PV005bgi',
       'PV006bgi', 'PV007bgi', 'PV008bgi', 'PV009', 'PV009bgi',
       'PV010bgi', 'PV011', 'PV012bgi', 'PV014', 'PV015bgi', 'PV016',
       'PV018bgi', 'PV020bgi', 'PV021bgi', 'PV022', 'PV023', 'PV024',
       'PV027', 'PV028', 'PV029', 'PV030', 'PV031', 'PV032', 'PV033',
       'PV034', 'PV035', 'PV036', 'PV037', 'PV041', 'PV042', 'PV046',
       'PV047', 'PV049', 'PV050', 'PV051', 'PV052', 'PV053', 'PV054',
       'PV055', 'PV056', 'PV057', 'PV058', 'PV059', 'PV060', 'PV061',
       'PV062', 'PV063', 'PV064', 'PV065', 'PV069', 'PV070', 'PV071',
       'PV072', 'PV074', 'PV075', 'PV076', 'PV078', 'PV079', 'PV080',
       'PV081', 'PV082', 'PV083', 'PV084', 'PV085', 'PV086', 'PV087',
       'PV088', 'PV089', 'PV091', 'PV092', 'PV094', 'PV095', 'PV096',
       'PV097', 'PV098', 'PV099', 'PV100', 'PV101', 'PV102', 'PV103',
       'PV104', 'PV105', 'PV106', 'PV107', 'PV108', 'PV109', 'PV110',
       'PV111', 'PV112', '

## Were all the virus detected in the pressence of a bacteria?

In [9]:
virus = db.conn.sql('SELECT * FROM D_virusHits').df()
bacteria = db.conn.sql('SELECT * FROM D_pabHits').df()
bacteria_positive_libraries = bacteria['library'].unique()
virus['is_in_bacteria'] = virus['library'].isin(bacteria_positive_libraries)
len(virus.groupby(['scientific_name'], as_index=False)['is_in_bacteria'].sum().query('is_in_bacteria != 0'))

109

## How many libraries has the site with most libraries?

In [10]:
pd.read_csv('output/metadata.site-library.csv', sep=';').value_counts('site')

site
L1    40
L3    35
L2    35
Q2    25
Q1    24
L4    23
E2    21
E4    19
Q3    16
E3    15
E1    14
Q4    13
M1     8
Z1     6
C2     6
C1     5
M3     5
H3     5
M4     4
M2     4
H2     4
H1     4
Z2     4
Name: count, dtype: int64

## What's the virus with wider range?

In [11]:
m = pd.merge(
    pd.read_csv("output/hits.virus.csv", sep=';'),
    pd.read_csv('output/metadata.site-library.csv', sep=';'),
    on='library'
)[['acronym', 'taxid', 'host_taxon']].drop_duplicates().value_counts('acronym').reset_index()

In [12]:
m

,acronym,count
0,CMV3,83
1,TMGMV,65
2,PZSV3,58
3,TMV,52
4,RuCMV,50
...,...,...
153,SbDV,1
154,CymRSV,1
155,CaMV,1
156,PCV2,1


In [13]:
m.query('count == 1')

,acronym,count
95,TYMV,1
96,BPeMV,1
97,BYSMV,1
98,BYDV_GPV,1
99,BYDV_KerII,1
...,...,...
153,SbDV,1
154,CymRSV,1
155,CaMV,1
156,PCV2,1


In [14]:
m = pd.merge(
    pd.read_csv("output/hits.bacteria.csv", sep=';').query('is_pab == True'),
    pd.read_csv('output/metadata.site-library.csv', sep=';'),
    on='library'
)[['scientific_name', 'taxid', 'host_taxon']].drop_duplicates().value_counts('scientific_name').reset_index()

In [15]:
len(m.query('count == 1'))

55

In [16]:
db.conn.close()